In [13]:
# ============================================================
# CELL 2 - ETL Process + SQLite Storage (hospital.db) + Logging
# ============================================================
import pandas as pd
import sqlite3
import logging

# --------------------
# Logging Setup
# --------------------
logging.basicConfig(
    filename='hospital_etl.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

try:
    logging.info("ETL process started")

    # --------------------
    # EXTRACT
    # --------------------
    df = pd.read_csv("hospital_patient_dataset.csv")
    logging.info(f"Extracted {len(df)} records from hospital_patient_dataset.csv")
    print(f"Extracted {len(df)} records")

    # --------------------
    # TRANSFORM
    # --------------------

    # Convert date columns
    df['AdmissionDate'] = pd.to_datetime(df['AdmissionDate'], errors='coerce')
    df['DischargeDate']  = pd.to_datetime(df['DischargeDate'],  errors='coerce')
    df['DateOfBirth']    = pd.to_datetime(df['DateOfBirth'],    errors='coerce')

    # Fill missing values
    df['FirstName']     = df['FirstName'].fillna('Unknown')
    df['LastName']      = df['LastName'].fillna('Unknown')
    df['Gender']        = df['Gender'].fillna('Unknown')
    df['ContactNumber'] = df['ContactNumber'].fillna('Not Available')
    df['Email']         = df['Email'].fillna('Not Available')
    df['Address']       = df['Address'].fillna('Unknown')
    df['Ward']          = df['Ward'].fillna('Unassigned')
    df['RoomNumber']    = df['RoomNumber'].fillna(0)
    df['Disease']       = df['Disease'].fillna('Under Diagnosis')
    df['DateOfBirth']   = df['DateOfBirth'].fillna(pd.to_datetime('1990-01-01'))

    # Standardise Gender  (M -> Male, F -> Female)
    df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female'})

    # Derived columns
    df['LengthOfStay'] = (df['DischargeDate'] - df['AdmissionDate']).dt.days
    df['LengthOfStay'] = df['LengthOfStay'].fillna(0)

    df['StayStatus'] = df['LengthOfStay'].apply(
        lambda x: 'Discharged' if x > 0 else 'Admitted / Ongoing'
    )

    logging.info("Transformation completed successfully")
    print("Transformation completed")
    print(df[['PatientID','FirstName','LastName','Gender','Disease','LengthOfStay','StayStatus']].to_string())

    # --------------------
    # LOAD  ->  hospital.db
    # --------------------
    conn = sqlite3.connect('hospital.db')
    df.to_sql('patients', conn, if_exists='replace', index=False)
    conn.close()

    logging.info("Data loaded into hospital.db (table: patients)")
    print("\nData loaded into hospital.db successfully")

except Exception as e:
    logging.error(f"ETL failed: {e}")
    print(f"ETL failed: {e}")

Extracted 20 records
Transformation completed
   PatientID FirstName  LastName   Gender                 Disease  LengthOfStay          StayStatus
0       P001      John       Doe     Male                 Typhoid           4.0          Discharged
1       P002      Jane   Unknown   Female                Diarrhea           1.0          Discharged
2       P003    Robert     Brown     Male               Pneumonia           5.0          Discharged
3       P004   Unknown     Davis   Female          Food Poisoning           0.0  Admitted / Ongoing
4       P005   Michael   Johnson     Male                Diabetes           0.0  Admitted / Ongoing
5       P006     Linda  Williams  Unknown            Hypertension           2.0          Discharged
6       P007     James    Miller     Male         Under Diagnosis           4.0          Discharged
7       P008     Sarah    Wilson   Female                  Asthma           3.0          Discharged
8       P009     David     Moore     Male             

In [14]:
# ============================================================
# CELL 3 - Scheduling
# ============================================================
import importlib.util
import sys

if importlib.util.find_spec("schedule") is None:
    print("schedule package not found, installing...")
    !{sys.executable} -m pip install schedule

import schedule
import time


def run_etl_job():
    print("Scheduler: Running ETL job...")
    logging.info("Scheduled ETL job triggered")
    try:
        df = pd.read_csv("hospital_patient_dataset.csv")

        # --------------------
        # TRANSFORM (same as ETL cell)
        # --------------------
        df['AdmissionDate'] = pd.to_datetime(df['AdmissionDate'], errors='coerce')
        df['DischargeDate']  = pd.to_datetime(df['DischargeDate'],  errors='coerce')
        df['DateOfBirth']    = pd.to_datetime(df['DateOfBirth'],    errors='coerce')

        df['FirstName']     = df['FirstName'].fillna('Unknown')
        df['LastName']      = df['LastName'].fillna('Unknown')
        df['Gender']        = df['Gender'].fillna('Unknown')
        df['ContactNumber'] = df['ContactNumber'].fillna('Not Available')
        df['Email']         = df['Email'].fillna('Not Available')
        df['Address']       = df['Address'].fillna('Unknown')
        df['Ward']          = df['Ward'].fillna('Unassigned')
        df['RoomNumber']    = df['RoomNumber'].fillna(0)
        df['Disease']       = df['Disease'].fillna('Under Diagnosis')
        df['DateOfBirth']   = df['DateOfBirth'].fillna(pd.to_datetime('1990-01-01'))

        df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female'})

        df['LengthOfStay'] = (df['DischargeDate'] - df['AdmissionDate']).dt.days
        df['LengthOfStay'] = df['LengthOfStay'].fillna(0)

        df['StayStatus'] = df['LengthOfStay'].apply(
            lambda x: 'Discharged' if x > 0 else 'Admitted / Ongoing'
        )

        # --------------------
        # LOAD into SQLite
        # --------------------
        with sqlite3.connect('hospital.db') as conn:
            df.to_sql('patients', conn, if_exists='replace', index=False)

        logging.info("Scheduled ETL job completed")
        print("Scheduler: ETL job done.")
    except Exception as e:
        logging.error(f"Scheduled ETL job failed: {e}")
        print(f"Scheduler: ETL job failed: {e}")


def configure_scheduler(production=False):
    schedule.clear()
    if production:
        schedule.every().day.at("02:00").do(run_etl_job)
        print("Scheduler configured: ETL runs daily at 02:00")
    else:
        schedule.every(1).minutes.do(run_etl_job)
        print("Scheduler configured: ETL test mode - every 1 minute")


def run_scheduler_loop():
    print("Starting scheduler loop. Press Ctrl+C to stop.")
    try:
        while True:
            schedule.run_pending()
            time.sleep(60)
    except KeyboardInterrupt:
        print("Scheduler loop stopped by user")


# Configure scheduler (production=False for notebook test). 
# Switch to production=True in deployment environment.
configure_scheduler(production=False)

# Run run_scheduler_loop() manually when ready in an always-on environment.
print("Call run_scheduler_loop() to begin scheduled execution.")

Scheduler configured: ETL test mode - every 1 minute
Call run_scheduler_loop() to begin scheduled execution.


In [15]:
# ============================================================
# CELL 4 - Logging & Monitoring Check
# ============================================================
import os

log_file = 'hospital_etl.log'

if os.path.exists(log_file):
    print("=== PIPELINE LOG ===")
    with open(log_file, 'r') as f:
        print(f.read())
else:
    print("No log file found yet. Run the ETL cell first.")

=== PIPELINE LOG ===
2026-03-27 17:22:10,045 - INFO - ETL process started
2026-03-27 17:22:10,052 - INFO - Extracted 20 records from hospital_patient_dataset.csv
2026-03-27 17:22:10,067 - INFO - Transformation completed successfully
2026-03-27 17:22:10,089 - INFO - Data loaded into hospital.db (table: patients)
2026-03-27 17:22:26,259 - INFO - ETL process started
2026-03-27 17:22:26,265 - INFO - Extracted 20 records from hospital_patient_dataset.csv
2026-03-27 17:22:26,281 - INFO - Transformation completed successfully
2026-03-27 17:22:26,302 - INFO - Data loaded into hospital.db (table: patients)
2026-03-27 17:23:52,216 - INFO - ETL process started
2026-03-27 17:23:52,220 - INFO - Extracted 20 records from hospital_patient_dataset.csv
2026-03-27 17:23:52,234 - INFO - Transformation completed successfully
2026-03-27 17:23:52,256 - INFO - Data loaded into hospital.db (table: patients)
2026-03-27 17:24:02,281 - INFO - ETL process started
2026-03-27 17:24:02,285 - INFO - Extracted 20 reco

In [16]:
# ============================================================
# CELL 5 - Report Generation
# ============================================================
import sqlite3
import pandas as pd

conn = sqlite3.connect('hospital.db')
df = pd.read_sql_query("SELECT * FROM patients", conn)

# --------------------
# REPORT 1: Patients per Ward
# --------------------
ward_report = df.groupby('Ward').agg(
    TotalPatients=('PatientID', 'count'),
    AvgStay=('LengthOfStay', 'mean')
).round(1)

ward_report.to_csv('ward_report.csv')
print("=== REPORT 1: Patients per Ward ===")
print(ward_report)

# --------------------
# REPORT 2: Disease Summary
# --------------------
disease_report = df.groupby('Disease').agg(
    TotalPatients=('PatientID', 'count'),
    AvgStay=('LengthOfStay', 'mean')
).round(1).sort_values('TotalPatients', ascending=False)

disease_report.to_csv('disease_report.csv')
print("\n=== REPORT 2: Disease Summary ===")
print(disease_report)

# --------------------
# REPORT 3: Gender Distribution
# --------------------
gender_report = df['Gender'].value_counts()
gender_report.to_csv('gender_report.csv')
print("\n=== REPORT 3: Gender Distribution ===")
print(gender_report)

# --------------------
# REPORT 4: Missing Data Report
# --------------------
missing_report = df.isnull().sum()
missing_report.to_csv('missing_report.csv')
print("\n=== REPORT 4: Missing Data Report ===")
print(missing_report)

conn.close()
print("\nAll reports generated successfully!")

=== REPORT 1: Patients per Ward ===
            TotalPatients  AvgStay
Ward                              
A                       6      2.0
B                       6      2.2
C                       4      4.5
D                       3      2.3
Unassigned              1      3.0

=== REPORT 2: Disease Summary ===
                        TotalPatients  AvgStay
Disease                                       
Asthma                              2      3.5
Dengue                              2      2.0
Diarrhea                            2      1.5
Diabetes                            2      0.0
Pneumonia                           2      5.5
Typhoid                             2      4.0
Chronic Kidney Disease              1      0.0
Anemia                              1      2.0
Food Poisoning                      1      0.0
Gastritis                           1      2.0
Hypertension                        1      2.0
Heart Disease                       1      7.0
Malaria                   